# Benchmark: Windowed Iterative SLSQP vs Full-Grid SLSQP

Compares two approaches for correcting negative Jacobian determinants:

- **Windowed (iterative):** Finds the worst pixel, extracts a small sub-window around the
  connected negative region, runs SLSQP on that window with frozen edges, then repeats.
  Window grows adaptively if needed.
- **Full-grid (single-shot):** Runs one SLSQP optimization over the entire H×W grid at once.
  All `2*H*W` displacement values are free variables with Jacobian constraints on every pixel.

The windowed approach should scale much better — the full-grid optimizer has to evaluate
constraints at every pixel on every SLSQP iteration, while the windowed approach only
touches small local regions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import time
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize, NonlinearConstraint

from dvfopt import (
    iterative_serial,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
    DEFAULT_PARAMS,
)
from dvfopt.jacobian.numpy_jdet import _numpy_jdet_2d
from dvfopt.core.objective import objective_euc
from benchmark_utils import plot_jac_heatmaps, plot_jdet_histogramsfrom benchmark_utils import get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter


In [ ]:
METHOD = "slsqp"
NOTEBOOK_NAME = "windowed-vs-fullgrid"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


## Full-grid SLSQP solver

In [ ]:
def fullgrid_slsqp(deformation_i, threshold=None, max_minimize_iter=None, verbose=0):
    """Correct negative Jacobians by running SLSQP on the entire grid at once."""
    threshold = threshold or DEFAULT_PARAMS["threshold"]
    max_minimize_iter = max_minimize_iter or DEFAULT_PARAMS["max_minimize_iter"]

    H, W = deformation_i.shape[-2:]
    pixels = H * W

    phi = np.zeros((2, H, W))
    phi[0] = deformation_i[-2]
    phi[1] = deformation_i[-1]
    phi_init = phi.copy()

    phi_flat = np.concatenate([phi[1].flatten(), phi[0].flatten()])
    phi_init_flat = phi_flat.copy()

    def jac_con(phi_xy):
        dx = phi_xy[:pixels].reshape(H, W)
        dy = phi_xy[pixels:].reshape(H, W)
        return _numpy_jdet_2d(dy, dx).flatten()

    constraints = [NonlinearConstraint(jac_con, threshold, np.inf)]

    result = minimize(
        lambda phi1: objective_euc(phi1, phi_init_flat),
        phi_flat,
        jac=True,
        constraints=constraints,
        options={"maxiter": max_minimize_iter, "disp": verbose >= 1},
        method="SLSQP",
    )

    phi[1] = result.x[:pixels].reshape(H, W)
    phi[0] = result.x[pixels:].reshape(H, W)
    return phi

## Configuration

In [ ]:
# Grid sizes to test.
# Full-grid SLSQP scales poorly — keep sizes small.
GRID_SIZES = [10, 15, 20, 30, 40, 60]

# Full-grid becomes extremely slow at larger sizes.
# Skip it above this threshold and only run windowed.
FULLGRID_MAX_SIZE = 30

BASE_SHAPE = (3, 1, 5, 5)
MAX_MAGNITUDE = 1.0   # low magnitude → ~5-8% negative pixels (localized)
SEED = 42

## Generate test fields

In [ ]:
base_dvf = generate_random_dvf(BASE_SHAPE, MAX_MAGNITUDE, SEED)

test_fields = {}
for size in GRID_SIZES:
    dvf = scale_dvf(base_dvf, (size, size))
    jac = jacobian_det2D(np.stack([dvf[-2, 0], dvf[-1, 0]]))
    n_neg = int((jac <= 0).sum())
    test_fields[size] = dvf
    print(f"  {size:>4d}x{size:<4d}  neg-Jdet: {n_neg:>5d}  "
          f"min Jdet: {jac.min():+.4f}  pixels: {size*size}")

## Run benchmarks

In [ ]:
results = {}

header = (f"{'Size':>10s}  {'Method':<12s}  {'Time (s)':>10s}  {'Neg init':>10s}  "
          f"{'Neg final':>10s}  {'Min Jdet':>10s}  {'L2 Error':>10s}  {'Vars':>8s}")
print(header)
print("-" * len(header))

for size in GRID_SIZES:
    dvf = test_fields[size]
    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())

    results[size] = {"n_neg_init": n_neg_init, "phi_init": phi_init, "jac_init": jac_init}

    # --- Windowed iterative ---
    t0 = time.perf_counter()
    phi_w = iterative_serial(dvf.copy(), verbose=0)
    t_windowed = time.perf_counter() - t0

    jac_w = jacobian_det2D(phi_w)
    results[size]["windowed"] = {
        "time": t_windowed,
        "neg": int((jac_w <= 0).sum()),
        "min_jdet": float(jac_w.min()),
        "l2": float(np.sqrt(np.sum((phi_w - phi_init) ** 2))),
        "phi": phi_w,
        "jac": jac_w,
    }
    print(f"{size:>4d}x{size:<4d}  {'windowed':<12s}  {t_windowed:>10.2f}  "
          f"{n_neg_init:>10d}  {results[size]['windowed']['neg']:>10d}  "
          f"{results[size]['windowed']['min_jdet']:>+10.6f}  "
          f"{results[size]['windowed']['l2']:>10.4f}  {'--':>8s}")

    # --- Full-grid ---
    if size <= FULLGRID_MAX_SIZE:
        n_vars = 2 * size * size
        t0 = time.perf_counter()
        phi_f = fullgrid_slsqp(dvf.copy(), verbose=0)
        t_full = time.perf_counter() - t0

        jac_f = jacobian_det2D(phi_f)
        results[size]["fullgrid"] = {
            "time": t_full,
            "neg": int((jac_f <= 0).sum()),
            "min_jdet": float(jac_f.min()),
            "l2": float(np.sqrt(np.sum((phi_f - phi_init) ** 2))),
            "n_vars": n_vars,
            "phi": phi_f,
            "jac": jac_f,
        }
        print(f"{'':>10s}  {'full-grid':<12s}  {t_full:>10.2f}  "
              f"{'':>10s}  {results[size]['fullgrid']['neg']:>10d}  "
              f"{results[size]['fullgrid']['min_jdet']:>+10.6f}  "
              f"{results[size]['fullgrid']['l2']:>10.4f}  {n_vars:>8d}")
    else:
        print(f"{'':>10s}  {'full-grid':<12s}  {'(skipped)':>10s}")

    print()

## Jacobian heatmaps — initial / windowed / full-grid

Each column is a grid size (where both methods ran).  Rows: initial, windowed result, full-grid result.

In [ ]:
sizes_both = [s for s in GRID_SIZES if "fullgrid" in results[s]]
plot_jac_heatmaps(
    [[results[s]["jac_init"][0] for s in sizes_both],
     [results[s]["windowed"]["jac"][0] for s in sizes_both],
     [results[s]["fullgrid"]["jac"][0] for s in sizes_both]],
    [f"{s}x{s}" for s in sizes_both],
    row_labels=["Initial", "Windowed", "Full-grid"],
    title="Jacobian Determinant — Initial / Windowed / Full-grid",
    figscale=3,
)
show_and_save(OUTPUT_DIR)


## Correction magnitude — windowed vs full-grid

Per-pixel displacement change `|phi_corrected - phi_init|`.  Full-grid distributes
the correction globally; windowed concentrates it around negative regions.

In [ ]:
n = len(sizes_both)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 5))
if n == 1:
    axes = axes[:, np.newaxis]

for i, size in enumerate(sizes_both):
    phi_init = results[size]["phi_init"]
    for row, (method, label) in enumerate([("windowed", "Windowed"), ("fullgrid", "Full-grid")]):
        ax = axes[row, i]
        diff = np.sqrt(((results[size][method]["phi"] - phi_init) ** 2).sum(axis=0))
        im = ax.imshow(diff, cmap="hot", origin="upper")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"{size}x{size}", fontsize=11)
        if i == 0:
            ax.set_ylabel(label, fontsize=11)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Per-pixel correction magnitude", fontsize=13)
plt.tight_layout()
show_and_save(OUTPUT_DIR)


## Jdet distributions — windowed vs full-grid

Overlaid histograms for the largest size where both methods ran.

In [ ]:
plot_jdet_histograms(
    [[("Initial", results[s]["jac_init"][0]),
      ("Windowed", results[s]["windowed"]["jac"][0]),
      ("Full-grid", results[s]["fullgrid"]["jac"][0])]
     for s in sizes_both],
    [f"{s}x{s}" for s in sizes_both],
    title="Jacobian Determinant Distribution Comparison",
    colors=["tab:gray", "tab:blue", "tab:red"],
    figscale=4,
)
show_and_save(OUTPUT_DIR)


---
## Scaling and metric plots

In [ ]:
# Collect data for plotting
sizes_both = [s for s in GRID_SIZES if "fullgrid" in results[s]]
sizes_all = GRID_SIZES

t_win_all = [results[s]["windowed"]["time"] for s in sizes_all]
t_win_both = [results[s]["windowed"]["time"] for s in sizes_both]
t_full_both = [results[s]["fullgrid"]["time"] for s in sizes_both]

l2_win_both = [results[s]["windowed"]["l2"] for s in sizes_both]
l2_full_both = [results[s]["fullgrid"]["l2"] for s in sizes_both]

min_w = [results[s]["windowed"]["min_jdet"] for s in sizes_both]
min_f = [results[s]["fullgrid"]["min_jdet"] for s in sizes_both]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# --- Runtime comparison ---
ax = axes[0, 0]
ax.plot(sizes_all, t_win_all, "o-", label="Windowed", color="tab:blue")
ax.plot(sizes_both, t_full_both, "s-", label="Full-grid", color="tab:red")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime Comparison")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Runtime comparison (log scale) ---
ax = axes[0, 1]
pixels_all = [s * s for s in sizes_all]
pixels_both = [s * s for s in sizes_both]
ax.loglog(pixels_all, t_win_all, "o-", label="Windowed", color="tab:blue")
ax.loglog(pixels_both, t_full_both, "s-", label="Full-grid", color="tab:red")
ax.set_xlabel("Total pixels")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime Scaling (log-log)")
ax.legend()
ax.grid(True, alpha=0.3)

# --- L2 error comparison ---
ax = axes[1, 0]
x = np.arange(len(sizes_both))
w = 0.35
ax.bar(x - w / 2, l2_win_both, w, label="Windowed", color="tab:blue", alpha=0.8)
ax.bar(x + w / 2, l2_full_both, w, label="Full-grid", color="tab:red", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{s}x{s}" for s in sizes_both])
ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation from Original")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# --- Min Jdet comparison ---
ax = axes[1, 1]
ax.bar(x - w / 2, min_w, w, label="Windowed", color="tab:blue", alpha=0.8)
ax.bar(x + w / 2, min_f, w, label="Full-grid", color="tab:red", alpha=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(0.01, color="gray", linewidth=0.8, linestyle="--", label="Threshold (0.01)")
ax.set_xticks(x)
ax.set_xticklabels([f"{s}x{s}" for s in sizes_both])
ax.set_ylabel("Min Jacobian determinant")
ax.set_title("Final Min Jdet (both should be > 0)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Windowed Iterative SLSQP vs Full-Grid SLSQP", fontsize=14)
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Speedup bar chart ---
speedups = [results[s]["fullgrid"]["time"] / results[s]["windowed"]["time"]
            for s in sizes_both]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f"{s}x{s}" for s in sizes_both], speedups, color="tab:green")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Speedup (full-grid time / windowed time)")
ax.set_title("Windowed Speedup over Full-Grid")
for bar, sp in zip(bars, speedups):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"{sp:.1f}x", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
show_and_save(OUTPUT_DIR)


## Summary

Key takeaways:

- **Full-grid** SLSQP treats *all* `2·H·W` displacement values as free variables with
  `H·W` Jacobian constraints. Each SLSQP iteration evaluates the full constraint Jacobian
  matrix, so cost grows rapidly with grid size.
- **Windowed** SLSQP only optimizes small local windows (typically 3×3 to ~15×15) around
  each negative region. Most of the grid is never touched. The number of SLSQP variables
  per solve is bounded regardless of grid size.
- Full-grid produces a **globally optimal** correction (lowest possible L2 error), while
  windowed finds a local solution that may have slightly higher L2 error.
- The windowed approach is the only viable option for real-world grids (64×91, 320×456, etc.).

In [ ]:
# --- Save results ---
if 'results' in dir() and isinstance(results, dict) and results:
    rows, cols = results_to_rows(results)
    save_results_csv(rows, cols, OUTPUT_DIR)
    summary = log_run_footer(summary, results)
    save_summary_json(summary, OUTPUT_DIR)
elif 'mag_results' in dir():
    rows, cols = results_to_rows(mag_results)
    save_results_csv(rows, cols, OUTPUT_DIR, name='results_magnitude')
    if 'density_results' in dir():
        rows2, cols2 = results_to_rows(density_results)
        save_results_csv(rows2, cols2, OUTPUT_DIR, name='results_density')
    combined = {**mag_results, **density_results} if 'density_results' in dir() else mag_results
    summary = log_run_footer(summary, combined)
    save_summary_json(summary, OUTPUT_DIR)
else:
    save_summary_json(summary, OUTPUT_DIR)
    print('No results dict found; saved summary only.')
